# Differential privacy with the privacy MCP tools

Apply Laplace / Gaussian mechanisms to real aggregate queries, track the privacy budget, and read off the (ε, δ) guarantee.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # repo root (kernel cwd)

import pandas as pd
from mcp_servers import privacy_tools as pt

# Ensure the cohort CSV exists (generated by 20_privacy_assessment).
if not Path('examples/privacy/clinical_cohort.csv').exists():
    from examples.privacy.clinical_cohort import build_cohort
    build_cohort(200, seed=7).to_csv('examples/privacy/clinical_cohort.csv', index=False)
cohort = pd.read_csv('examples/privacy/clinical_cohort.csv')
counts = cohort.groupby(cohort['admission_date'].str[:7]).size().sort_index()
print(counts)

In [ ]:
import json
dp = json.loads(pt.apply_laplace_dp(counts.tolist(), epsilon=0.5, sensitivity=1.0, seed=7))
print('scale =', dp['scale'])
print('noisy =', [round(v, 1) for v in dp['noisy_values']])
print('guarantee =', dp['privacy_guarantee'])
print(pt.dp_guarantee_summary(0.5, 1e-6))

In [ ]:
print(pt.dp_privacy_budget_report([
    {'epsilon': 0.5, 'delta': 0.0, 'description': 'admission histogram'},
    {'epsilon': 0.3, 'delta': 0.0, 'description': 'mean visit amount'},
    {'epsilon': 0.2, 'delta': 0.0, 'description': 'per-condition counts'},
]))

print('Total ε budget spent: 1.0 of a recommended 1.0 max.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
dp = json.loads(pt.apply_laplace_dp(counts.tolist(), epsilon=0.5, sensitivity=1.0, seed=7))
x = np.arange(len(counts))
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(x - 0.2, counts.values, width=0.4, label='original', color='#e05b5b')
ax.bar(x + 0.2, dp['noisy_values'], width=0.4, label='Laplace ε=0.5', color='#35c4b6')
ax.set_xticks(x); ax.set_xticklabels(counts.index, rotation=45, fontsize=8)
ax.set_ylabel('admissions / month'); ax.legend()
ax.set_title('Original vs differentially-private histogram')